# Modern Application Development – I: Comprehensive Lecture Notes  
**Professor Nitin Chandrachoodan, IIT Madras**  
**Week 9: Security – Access Control, Mechanisms, Sessions, HTTPS & Logging**

---

## Table of Contents
1. [Access Control: Fundamentals and Models](#1-access-control-fundamentals-and-models)
2. [Security Mechanisms for Web Applications](#2-security-mechanisms-for-web-applications)
3. [Session Management: Overcoming HTTP Statelessness](#3-session-management)
4. [HTTPS: Securing the Communication Channel](#4-https-securing-the-communication-channel)
5. [Logging and Monitoring](#5-logging-and-monitoring)

---

## 1. Access Control: Fundamentals and Models

### 1.1 What is Access Control?
**Access control** is the selective restriction of access to resources. In computing, it means ensuring that only authorised users or systems can read, modify, delete, or execute specific data or functionality. It answers the question: *Who can do what to which resource?*

In a web application, not all functionality and data should be exposed to every visitor. For example:
- A student should see only their own grades, not the grades of their classmates.
- A customer on an e‑commerce site should see their own shopping cart and order history, not those of other customers.
- An administrator may need the ability to add new courses or reset passwords, a power that regular users must not have.
- Even read‑only access must be controlled: financial records, medical data, and personal emails must be visible only to their legitimate owners.

Access control operates at various levels of granularity. It can be as coarse as “this page is only for logged‑in users” or as fine‑grained as “this specific field in this record can be edited only by the owner during business hours”.

### 1.2 Types of Access
Access is not just a binary “allow/deny”. There are different modes:
- **Read:** View the data.
- **Write:** Modify existing data.
- **Create:** Add new data.
- **Delete:** Remove data.
- **Execute:** Run a program or script.

A user may be allowed to read a document but not modify it, or to add a comment but not delete others’ comments. A well‑designed access control system distinguishes these modes and enforces them individually.

### 1.3 Discretionary vs. Mandatory Access Control
**Discretionary Access Control (DAC):**
- The owner of a resource (e.g., a file’s creator) decides who else can access it and in what mode.
- Example: On a Linux filesystem, the file’s owner can use `chmod` to grant read, write, or execute permissions to the group or to everyone.
- In email, a recipient can forward the message to someone else—the original sender does not control that redistribution.
- DAC is flexible and easy to use, but it can lead to unintended information leakage if owners are careless or if an attacker gains an owner’s credentials.

**Mandatory Access Control (MAC):**
- The system enforces a central policy that users cannot override. Even if you own a file, you cannot grant access to someone who is not cleared by the policy.
- Commonly used in military and intelligence environments where data is classified (Top Secret, Secret, Confidential) and users have corresponding clearances. A user with “Secret” clearance cannot read “Top Secret” documents even if the owner tries to share them.
- MAC is much stricter and harder to bypass, but it is also more rigid and complex to administer.

Most web applications employ discretionary access control, where application logic (e.g., “only the user who created this post can edit it”) acts as the gatekeeper. The principle remains: the data owner has significant control within the bounds set by the application.

### 1.4 Role‑Based Access Control (RBAC)
In many systems, it is impractical to assign permissions to every individual user. Instead, permissions are assigned to **roles**, and users are assigned to those roles.

**Role examples in a university:**
- **Student:** Can view their own grades, register for courses.
- **Teacher:** Can view the list of students in their course, enter marks, post announcements.
- **Head of Department:** Inherits all teacher permissions and can additionally view aggregate performance data for the department, approve new course offerings.
- **Admin:** Can create users, assign roles, change system settings.

A single user can hold multiple roles. A person might be a teacher and also the faculty advisor for a sports club, having additional permissions related to club facilities that are unrelated to their academic role.

**Benefits of RBAC:**
- **Simplified administration:** When the HOD changes, only the role assignment is updated; all permissions tied to “HOD” automatically transfer.
- **Least privilege enforcement:** A user gets only the permissions of their assigned roles.
- **Role hierarchies:** Permissions can be inherited (e.g., HOD ⊇ Teacher ⊇ Student).

RBAC can be implemented in web applications by checking the user’s role(s) in the controller before executing a sensitive operation.

### 1.5 Attribute‑Based Access Control (ABAC)
ABAC goes beyond static roles by evaluating **attributes** of the user, the resource, the action, and the environment. Policies are expressed as logical combinations of these attributes.

**Example policy:** “A bank employee can view ledger entries **only if** the current time is between 9 AM and 5 PM on a working day **and** they are connecting from an internal network.”

Attributes can include:
- **User attributes:** department, clearance level, job title.
- **Resource attributes:** classification level, owner, creation date.
- **Action attributes:** read, write, delete.
- **Environmental attributes:** time of day, client IP address, authentication strength.

ABAC provides extreme flexibility but requires a powerful policy engine and careful definition of attributes to avoid conflicts and performance problems.

### 1.6 Permissions vs. Policies
- **Permissions** are simple, static rules: “User 42 can read file X.”
- **Policies** combine multiple conditions: “Allow read if user.department = ‘Finance’ AND resource.type = ‘report’ AND time.hour < 17.”

Policies can be layered. A base policy might grant all employees read access to a certain document store, while a stricter policy restricts some documents to specific roles. The final access decision is the combination of all applicable policies (usually the most restrictive wins).

### 1.7 The Principle of Least Privilege
**Least privilege** states that every entity (user, process, program) should be given the *minimum* access rights necessary to perform its intended function. No more.

**Examples:**
- A delivery person needs your address to deliver a package; they do not need a key to your house.
- On a Linux server, regular users can read shared libraries (`/usr/lib`) but cannot modify them. Only `root` can install or update system‑wide software.
- A web application’s database user account should only have the SQL privileges it needs (e.g., `SELECT`, `INSERT`, `UPDATE` on specific tables), not `DROP DATABASE`.

**Benefits:**
- **Better security:** If a component is compromised, the attacker’s access is limited.
- **Better stability:** Accidental modifications to critical files are prevented.
- **Easier deployment and auditing:** With well‑defined, minimal permissions, it is easier to understand what a component does and to replicate configurations.

**Challenges:** In practice, strictly adhering to least privilege can be difficult. Some software (including web frameworks) may need write access to certain directories for uploads or caching, forcing a compromise. Nevertheless, it remains a guiding principle.

### 1.8 Privilege Escalation
**Privilege escalation** is the act of gaining higher access rights than normally granted. It can be a legitimate, controlled action (like using `sudo` on Linux) or a malicious exploit.

**Legitimate escalation:** On Linux, `sudo` allows an authorised user to execute a single command with superuser privileges. The system logs every `sudo` invocation. The user is expected to `sudo` only when absolutely necessary and return to normal privileges immediately after.

**Malicious escalation:** An attacker exploits a software bug (e.g., buffer overflow) to execute code with elevated privileges. Web applications must be carefully coded to prevent such exploits (input validation, proper session handling, etc.).

The lecture advises that administrative actions in a web app should follow a similar pattern: the admin should log in to a normal account for daily use and only escalate to an admin role (perhaps via a separate admin dashboard with re‑authentication) when needed. Persistent admin sessions increase the attack surface.

### 1.9 Enforcing Access Control in Web Applications
Access control can be enforced at multiple layers:
- **Hardware:** Physical locks, biometric scanners, smart cards.
- **Operating System:** File permissions, memory protection, process isolation.
- **Database:** User accounts with table/column‑level privileges.
- **Application:** The custom logic that decides whether the current request should be allowed to proceed.

In a web app following MVC, the **controller** is the ideal place to enforce access control. Before fetching data from the model or rendering a view, the controller can check:
- Is the user authenticated?
- Does the user have the required role or permissions?
- Are there any additional attribute constraints (e.g., rate limits, IP restrictions)?

In Flask, this is typically implemented using **decorators** that wrap route functions. For example, a `@login_required` decorator can redirect unauthenticated users to a login page, and a `@admin_required` decorator can return a 403 Forbidden for non‑admins. We will see concrete examples later.

---

## 2. Security Mechanisms for Web Applications

### 2.1 What NOT to Do: Security by Obscurity
**Security by obscurity** relies on keeping the system’s design or implementation secret. For example, running an admin interface on a non‑standard port (e.g., 12345) and hoping attackers won’t find it. This is **not** real security.

- Port scanners can rapidly discover open ports.
- Information leaks (e.g., someone accidentally sharing a URL) instantly break the “obscurity”.
- Once discovered, there are no further defences.

Real security mechanisms do not depend on secrecy of the mechanism itself (Kerckhoffs’s principle). They depend on secrets (keys, passwords) that are small, changeable, and protectable.

### 2.2 Host‑Based Access Control
Restricting access based on the client’s IP address or hostname. For example, an SSH server might only accept connections from a specific jump host, or a web app’s admin area might only be reachable from the internal office network.

**Use cases:**
- Restricting administrative interfaces to an internal VPN.
- Geo‑blocking (denying requests from certain countries).

**Weaknesses:**
- IP addresses can be spoofed (though difficult for TCP/HTTPS).
- If the trusted intermediate host is compromised, the attacker gains access.
- Not suitable as the sole authentication mechanism for sensitive operations.

It is a useful **additional** layer (defence in depth) but should never be the only protection.

### 2.3 Login Mechanisms and Password Storage
The most common web authentication method: a form asking for username and password.

**Critical rule:** **Never store passwords in plain text.** If the server is breached, all passwords are immediately exposed. Given that users often reuse passwords, this compromises not only your application but also the user’s accounts elsewhere.

**Solution:** Store a **cryptographic hash** of the password (e.g., bcrypt, scrypt, Argon2). A hash function is a one‑way transformation: it is easy to compute `hash(password)` but computationally infeasible to reverse it. During login, the server hashes the provided password and compares it to the stored hash.

To thwart pre‑computed rainbow‑table attacks, a unique random **salt** is added to each password before hashing. Modern frameworks (like Flask’s Werkzeug, Django) provide built‑in secure password hashing utilities.

### 2.4 API Keys and Tokens
For machine‑to‑machine communication (e.g., a script pulling data from your API), interactive login forms are not feasible. Instead, a **token** or **API key** is used.

- The user registers and receives a long, randomly generated string (the token).
- On every API request, the token is sent in a header (e.g., `Authorization: Bearer <token>`).
- The server looks up the token, verifies it, and identifies the associated user/permissions.

**Advantages:**
- Tokens can be created and revoked independently of the user’s main password.
- They can have scoped permissions (e.g., read‑only, limited to specific endpoints).
- If a token is compromised, it can be revoked without affecting the user’s password.

Tokens must be transmitted over HTTPS and stored securely by the client.

### 2.5 HTTP Basic Authentication
Defined in the HTTP specification, Basic Authentication is a simple challenge‑response mechanism.

**Flow:**
1. Client requests a protected resource.
2. Server responds with `401 Unauthorized` and a `WWW-Authenticate: Basic realm="..."` header. The `realm` is a string describing the protected area.
3. Browser displays a built‑in login dialog. The user enters username and password.
4. The browser concatenates `username:password`, encodes them in **Base64**, and sends a new request with the header `Authorization: Basic <encoded_string>`.
5. Server decodes the string, extracts username and password, verifies them, and serves the resource.

**Major weaknesses:**
- **Base64 is encoding, not encryption.** Anyone who intercepts the request can trivially decode the username and password. Therefore, Basic Authentication is completely insecure unless the entire connection is encrypted with HTTPS.
- **No built‑in logout.** The browser caches the credentials and sends them with every request to the same realm until the browser is closed. There is no standard server‑initiated way to force the browser to “forget” the credentials.
- **Password seen by server in plain text.** The server receives the actual password, which is undesirable even over HTTPS (a compromised server could log it).

**Conclusion:** Basic Auth is simple but should only be used over HTTPS and even then, more modern, token‑based schemes are generally preferable.

### 2.6 HTTP Digest Authentication
Digest Authentication improves on Basic Auth by never sending the password in the clear.

**Flow:**
1. Client requests resource.
2. Server responds with `401` and a `WWW-Authenticate: Digest` header containing a **nonce** (a random number used once), the realm, and other parameters.
3. Client computes a hash:
   - `HA1 = MD5(username:realm:password)`
   - `HA2 = MD5(method:uri)`
   - `response = MD5(HA1:nonce:HA2)`
4. Client sends back the username, nonce, response, and other values in the `Authorization` header.
5. Server, which knows the password (or its hash), performs the same computation and compares the response.

**Advantages over Basic:**
- The password is never transmitted; only a hash involving a nonce is sent.
- The nonce prevents replay attacks (the same response cannot be reused).
- The method and URI are included, preventing the request from being forwarded to a different path.

**Weaknesses:**
- The server must have access to the plain text password (or a password equivalent) to compute HA1, which means passwords cannot be stored using strong salted hashes like bcrypt. This is a significant security drawback.
- MD5 is considered cryptographically broken, though the specific construction in Digest Auth still provides reasonable protection in practice. Modern systems generally prefer form‑based login over HTTPS.

### 2.7 Client Certificates
Instead of (or in addition to) passwords, a client can present a **digital certificate** to prove its identity. The certificate contains a public key and is signed by a Certificate Authority (CA). The server can challenge the client to prove ownership of the corresponding private key.

**Advantages:**
- Very strong authentication; private key can be stored in hardware (smart card, TPM) and never leaves the device.
- No passwords to remember or be phished.

**Challenges:**
- Distributing and managing client certificates is complex.
- Users must keep their private key secure; if the device is stolen, the key is compromised.
- Not widely used for consumer web applications, but common in enterprise, VPNs, and high‑security government systems.

### 2.8 Form‑Based Authentication and Best Practices
The most common web login method uses an HTML form (`<form method="POST">`) with username and password fields.

**Critical rules:**
- **Always use `POST`**, never `GET`. `GET` requests have the form data in the URL, which is logged by servers, proxies, and browser history, exposing passwords.
- **Always use HTTPS.** Without it, the form data is sent in plain text over the network.
- **Implement CSRF protection.** A Cross‑Site Request Forgery (CSRF) token is a hidden form field containing a random value that the server generates and checks on submission. This prevents an attacker from tricking a logged‑in user’s browser into unknowingly submitting a malicious form.
- **Validate on the server.** Client‑side JavaScript validation is a UX convenience; server‑side validation is a security necessity.
- **Use a secure password hashing algorithm** (bcrypt, Argon2) on the server.

### 2.9 API Security Tokens (Bearer Tokens)
As introduced earlier, tokens are the preferred method for API authentication. They avoid sending the password on every request. A popular standard is **JWT (JSON Web Token)**, where the token itself contains cryptographically signed claims (e.g., user ID, expiry). The server can verify the token without a database lookup, improving scalability. However, token revocation becomes a challenge (blacklists or short expiry times are used).

---

## 3. Session Management: Overcoming HTTP Statelessness

### 3.1 The Need for State
HTTP is a stateless protocol. By default, each request is independent; the server does not remember previous requests from the same client. However, almost all meaningful web applications require state: a user logs in, adds items to a cart, navigates through a multi‑step form. **Sessions** are the mechanism to maintain state across HTTP requests.

### 3.2 Cookies
A **cookie** is a small piece of data sent by the server to the browser via the `Set-Cookie` response header. The browser stores the cookie and automatically sends it back in every subsequent request to the same domain using the `Cookie` header.

A cookie can contain:
- A name‑value pair.
- An **expiration** date (session cookies are deleted when the browser closes; persistent cookies survive).
- A **domain** and **path** restricting where the cookie is sent.
- **Secure** flag – cookie is only sent over HTTPS.
- **HttpOnly** flag – cookie is inaccessible to JavaScript (mitigates XSS theft).

Cookies are the foundation of web session management. The server can either:
- Store all session data directly in the cookie (after encrypting and signing it) – **client‑side sessions**.
- Store only a random session ID in the cookie, with the actual data kept in server‑side storage – **server‑side sessions**.

### 3.3 Client‑Side Sessions (Flask Example)
Flask provides a built‑in **signed cookie‑based session**. Data is stored in the cookie itself, but it is cryptographically signed using a secret key known only to the server.

**Setup:**
```python
app.secret_key = b'_5#y2L"F4Q8z\n\xec]/'  # MUST be kept secret!
```

**Using the session:**
The `session` object behaves like a dictionary:
```python
from flask import session

@app.route('/')
def index():
    if 'username' in session:
        return f'Logged in as {session["username"]}'
    return 'You are not logged in'

@app.route('/login', methods=['GET', 'POST'])
def login():
    if request.method == 'POST':
        session['username'] = request.form['username']
        return redirect(url_for('index'))
    return '''
        <form method="post">
            <input name="username">
            <input type="submit" value="Login">
        </form>'''
```
- Data is signed, so the user cannot tamper with it without invalidating the signature.
- Data is **not encrypted** by default; sensitive information should be stored on the server or the session cookie should be encrypted.
- The secret key must be truly secret, never committed to version control, and randomly generated.

### 3.4 Server‑Side Sessions
For larger amounts of data or sensitive information, server‑side sessions are preferable. The cookie contains only a random session ID. The server stores the session data in a database, filesystem, or (most commonly) a fast in‑memory store like **Redis** or **Memcached**.

**Flask extensions** like Flask‑Session can transparently switch the session backend to Redis, databases, etc. This approach keeps sensitive data off the client and allows the server to invalidate sessions at will (by deleting the session data).

### 3.5 Security Issues with Sessions and Cookies
**Cookie theft / Session hijacking:**
If an attacker steals a user’s session cookie (e.g., via XSS, packet sniffing on open WiFi, or physical access to an unlocked machine), they can impersonate that user. Mitigations:
- Mark cookies as `HttpOnly` (prevents JavaScript access).
- Mark cookies as `Secure` (only sent over HTTPS).
- Set short session lifetimes and provide logout functionality.
- Regenerate the session ID after login to prevent session fixation.

**Cross‑Site Request Forgery (CSRF):**
An attacker tricks a user’s browser into making an unwanted request to a site where the user is authenticated. For example, an `<img>` tag on `evil.com` could trigger a `GET` request to `bank.com/transfer?to=attacker&amount=1000`. If the user is logged into the bank, the browser sends the session cookie, and the transfer might succeed.

Mitigations:
- Use **CSRF tokens**: unique, unpredictable values embedded in forms and validated on the server.
- Use the `SameSite` cookie attribute (`Strict` or `Lax`) to prevent the browser from sending cookies on cross‑site requests.
- Require re‑authentication for sensitive actions.

### 3.6 Enforcing Authentication with Decorators
Flask allows decorating route functions to require authentication. A typical pattern using **Flask‑Login** extension:

```python
from flask_login import login_required, current_user

@app.route('/profile')
@login_required
def profile():
    return render_template('profile.html', user=current_user)
```
- The `@login_required` decorator runs before `profile()`. If the user is not authenticated, it redirects them to the login page.
- `current_user` is a proxy that provides access to the logged‑in user’s object.

This pattern can be extended to role‑based checks:
```python
def admin_required(f):
    @wraps(f)
    def decorated_function(*args, **kwargs):
        if not current_user.is_admin:
            abort(403)
        return f(*args, **kwargs)
    return decorated_function

@app.route('/admin')
@login_required
@admin_required
def admin_dashboard():
    ...
```

**Logout** clears the session:
```python
@app.route('/logout')
@login_required
def logout():
    logout_user()
    return redirect(url_for('index'))
```
This removes the user’s ID from the session (or deletes the server‑side session), effectively ending the session.

---

## 4. HTTPS: Securing the Communication Channel

### 4.1 The Problem with Plain HTTP
In plain HTTP, all data travels over the network as readable text. Anyone with access to the physical network, the WiFi access point, or any intermediate router can **eavesdrop** on the traffic. They can also **modify** requests and responses in transit (Man‑in‑the‑Middle attack). This means passwords, cookies, personal data—everything—is exposed.

### 4.2 HTTPS = HTTP over TLS
**HTTPS** wraps HTTP inside **TLS (Transport Layer Security)**, previously called SSL (Secure Sockets Layer). TLS provides:
- **Encryption:** The data is encrypted, so eavesdroppers see only meaningless bytes.
- **Integrity:** Any tampering with the data in transit is detected.
- **Authentication:** The client can verify that the server truly is who it claims to be (e.g., `google.com`).

### 4.3 How Encryption is Established (High‑Level)
TLS uses a combination of **asymmetric** (public‑key) and **symmetric** cryptography.

1. **Handshake:** The client and server agree on cipher suites and exchange random numbers.
2. **Server authentication:** The server presents its digital certificate. The client verifies the certificate’s validity against a trusted Certificate Authority (CA).
3. **Key exchange:** Using asymmetric cryptography (e.g., Diffie‑Hellman), the client and server securely compute a shared **session key** without ever sending it over the wire in a way an eavesdropper could decipher.
4. **Symmetric encryption:** All subsequent data is encrypted with the session key using fast symmetric ciphers (e.g., AES).

Even if the entire handshake is observed, an attacker cannot derive the session key because of the mathematical properties of the key exchange.

### 4.4 Certificates and the Chain of Trust
The crucial part is verifying that the server is not an impostor. This is achieved through **digital certificates** and a **Public Key Infrastructure (PKI)**.

- **Server Certificate:** A document that binds a public key to a domain name (e.g., `mail.google.com`). It contains the domain, the public key, the issuer, validity period, and is digitally signed by the issuer.
- **Certificate Authority (CA):** A trusted third party that issues certificates after validating the applicant’s control of the domain.
- **Chain of Trust:**
  - Your browser/OS comes with a pre‑installed set of **trusted root CAs** (e.g., “USERTrust”, “ISRG Root X1” (Let’s Encrypt), “DigiCert Global Root”).
  - The root CA signs an **intermediate CA** certificate.
  - The intermediate CA signs the **server certificate**.
  - When you connect to `google.com`, the server sends its certificate along with the intermediate certificate. Your browser verifies the signatures up the chain until it reaches a trusted root.

If any link in the chain is invalid (expired, signature mismatch, wrong domain), the browser displays a security warning.

**Example:** `mail.google.com` → signed by `GTS CA` (Google Trust Services) → signed by `GlobalSign Root CA` (or Google’s own root, depending on the chain).

### 4.5 Wildcard Certificates
A **wildcard certificate** covers all subdomains of a domain. For example, a certificate for `*.iitm.ac.in` is valid for `www.iitm.ac.in`, `courses.iitm.ac.in`, `mail.iitm.ac.in`, etc., but **not** for `sub.domain.iitm.ac.in`. Wildcard certificates simplify management when a single entity controls many subdomains.

The lecture shows the certificate for `iitm.ac.in` issued by Sectigo, chaining up to USERTrust root.

### 4.6 Potential Certificate Problems
- **Expired certificates:** Browsers reject expired certificates, breaking the site.
- **Stolen private keys:** If a server’s private key is stolen, the attacker can impersonate the server. The CA must revoke the certificate, and browsers check Certificate Revocation Lists (CRLs) or use OCSP (Online Certificate Status Protocol) to see if a certificate is still valid.
- **Compromised CA:** If a root or intermediate CA’s private key is stolen, all certificates issued by that CA become untrustworthy. This has happened in the past (e.g., DigiNotar in 2011), leading to the CA being removed from browser trust stores.
- **DNS hijacking:** If DNS is compromised, the user might be directed to an attacker’s IP. The certificate would then be invalid for that domain, causing a browser warning. However, many users ignore warnings, which is dangerous.

### 4.7 Impact of HTTPS
**Positives:**
- Confidentiality and integrity of data in transit.
- User trust (padlock icon).
- Enables use of modern web features like Service Workers, Geolocation, and HTTP/2 (which browsers often require HTTPS for).

**Negatives:**
- **Performance overhead:** TLS handshake adds latency (one extra round trip) and CPU load for encryption. However, modern hardware acceleration and session resumption (abbreviated handshakes) mitigate this.
- **Caching challenges:** Because the traffic is encrypted, intermediate proxies (e.g., corporate caches, ISP caches) cannot inspect the content to cache it. This reduces the effectiveness of shared caches. CDN providers offer HTTPS‑aware caching where they terminate the TLS connection at their edge and have the plaintext internally.

Despite the downsides, the security benefits are so overwhelming that the entire web is moving towards HTTPS everywhere. Organisations like Let’s Encrypt provide free, automated certificates, making HTTPS accessible to all.

---

## 5. Logging and Monitoring

### 5.1 Why Logging is Essential
**Logging** is the practice of recording events and data about an application’s operation. It serves multiple purposes:
- **Debugging:** Understanding why an error occurred.
- **Performance monitoring:** Tracking response times, database query counts.
- **Security auditing:** Detecting and investigating suspicious activity (e.g., repeated failed logins, access to unusual URLs).
- **Usage analytics:** Understanding user behaviour to improve the application.

Logging is a developer’s eyes and ears into a running application, especially on remote servers.

### 5.2 Web Server‑Level Logs
Web servers like Apache and Nginx automatically generate **access logs** and **error logs**.

**Typical access log format (Combined Log Format):**
```
127.0.0.1 - - [25/Jul/2026:13:55:36 +0530] "GET /index.html HTTP/1.1" 200 2326
```
- Client IP address.
- Authenticated user (often `-`).
- Timestamp.
- HTTP method, URL, protocol version.
- Status code (`200` for success, `404` for not found, `500` for server error).
- Size of the response body.

Server logs can reveal:
- **Broken links:** Frequent `404` errors for a specific URL.
- **Brute‑force attacks:** Many `POST /login` requests in a short time, often returning `401`.
- **Scanning for vulnerabilities:** Requests like `GET /wp-admin` or `GET /.env` that indicate an attacker probing for known weaknesses.

**Error logs** capture server crashes, configuration problems, and application errors that percolate up.

### 5.3 Application‑Level Logging
Relying only on server logs is insufficient for debugging application logic, because the server only sees the HTTP surface. Application logging, using Python’s built‑in `logging` module, allows granular recording.

**Basic Python logging setup:**
```python
import logging
logging.basicConfig(filename='app.log', level=logging.INFO)

@app.route('/')
def index():
    logging.info('Index page accessed')
    # ...
```
Log levels (DEBUG, INFO, WARNING, ERROR, CRITICAL) allow filtering.

**What to log at the application level:**
- Entry and exit of controller functions.
- Input parameters (after sanitising passwords!).
- Database query times.
- Exceptions and stack traces.
- Authentication events (logins, logouts, failed attempts).
- Any security‑relevant decisions (access denied, role checks).

**Never log sensitive data:** passwords, credit card numbers, full session tokens. Logs are often stored with less protection and may be accessed by many people.

### 5.4 Log Management: Rotation and Storage
Logs can grow extremely fast. Unchecked, they can fill up the server’s disk and crash the application. **Log rotation** is essential.

- **Time‑based rotation:** Start a new log file every day/hour, keep only the last N files.
- **Size‑based rotation:** When the current log reaches a certain size (e.g., 10 MB), rename it and start a new one.
- Tools like Linux’s `logrotate` or Python’s `RotatingFileHandler` / `TimedRotatingFileHandler` automate this.

### 5.5 From Logs to Insights: Time‑Series Analysis
Logs are inherently **time‑series data**—every entry has a timestamp. Analysing this data can reveal trends: peak usage times, error rate spikes, slow response degradation.

Dedicated time‑series databases (InfluxDB, Prometheus) and log aggregation tools (ELK Stack: Elasticsearch, Logstash, Kibana; or Grafana Loki) are designed to ingest, index, and visualise large volumes of log data. They allow:
- Full‑text search across logs.
- Real‑time dashboards of request rates, error percentages.
- Alerting when thresholds are crossed (e.g., `500` error rate > 1%).

For cloud‑hosted applications (Google App Engine, AWS Elastic Beanstalk), the platform provides built‑in logging and monitoring services (Stackdriver, CloudWatch). Using these services is highly recommended.

### 5.6 Summary of Security Best Practices
The lecture concludes with a synthesis of security wisdom for the app developer:

1. **Never trust user input.** Validate and sanitise everything on the server.
2. **Use established frameworks.** They handle many security pitfalls (CSRF protection, password hashing, session management) that are hard to get right manually.
3. **Enforce least privilege.** Design your database users, file permissions, and application roles with minimal access.
4. **Protect secrets.** Never hard‑code API keys, database passwords, or Flask secret keys in source code. Use environment variables or a secrets manager.
5. **Use HTTPS everywhere.**
6. **Log extensively, but securely.** Log enough to diagnose and audit, but never log secrets.
7. **Monitor and analyse logs.** Security is not a one‑time setup; it requires continuous vigilance. Review logs for anomalies, keep dependencies updated, and be prepared to respond to incidents.

Security is a vast, evolving field. The goal of this lecture is not to make you a security expert, but to instil a security *mindset*: always question how your application could be attacked, and apply the principle of defence in depth.